# 03 - Causal Audits of ORION Neural Tokens

**Question:** Which temporal token intervals does a downstream score actually depend on?

This notebook uses a deliberately transparent scorer so the intervention mechanics are easy to inspect. Replace it with an encoder/decoder score in real experiments.

**Evidence tier:** scientific synthetic for this toy batch. Do not generalize it to a real dataset claim.


In [ ]:
import numpy as np
from orion.contracts import NeuroTokenBatch
from neuros_mechint import EvidenceTier
from neuros_mechint.integrations.orion import temporal_window_audit


In [ ]:
batch = NeuroTokenBatch(
    token_ids=np.array([1, 2, 8, 9, 2, 1], dtype=np.int64),
    timestamps_ns=np.array([0, 10, 20, 30, 40, 50], dtype=np.int64) * 1_000_000,
    side_features={'population_rate': np.array([1, 1, 4, 4, 1, 1], dtype=np.float32)},
)

# Toy behavior: high IDs in the middle of the sequence matter most.
def scorer(tokens):
    return float(np.asarray(tokens.token_ids).sum())

result = temporal_window_audit(
    batch,
    scorer,
    window_ns=20_000_000,
    evidence_tier=EvidenceTier.SCIENTIFIC_SYNTHETIC,
    seed=7,
)
[(effect.target, effect.effect) for effect in result.effects]


## What this establishes

A large effect means the chosen score is sensitive to the specified token edit. It does **not** prove that the masked interval contains a unique semantic concept or identify an internal model circuit.

## Controls to add for real work

- matched random windows;
- deterministic within-window shuffles;
- multiple mask/replacement baselines;
- held-out sessions or subjects;
- equal downstream model budgets across tokenizers;
- replication across seeds and checkpoints.


## Extension

Run the same downstream task with ORION event, binned-count, relative-ISI, burst, synchrony, VQ-motif, and assembly tokenizers. Compare not only decoding accuracy, but the stability and sparsity of the causal token-window map. A tokenizer that is compact yet destroys transferable causal structure may not be a good foundation-model representation.
